## Exercice 1 : Analyse exploratoire des données (EDA)

Dans cette section, nous chargeons le jeu de données sur les maladies cardiaques pour examiner sa structure. Nous séparons ensuite la colonne cible des caractéristiques, encodons les variables catégorielles, et divisons le tout en ensembles d'entraînement et de test avant d'appliquer une standardisation.

In [83]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Chargement du fichier
chemin = 'Heart Disease Prediction Dataset/dataset_heart.csv'
df = pd.read_csv(chemin)

print("--- Aperçu des données ---")
print(df.head())

# 2. Séparation des caractéristiques et de la cible
nom_cible = 'heart disease'
X = df.drop(columns=[nom_cible])

# LA CORRECTION : Aligner les classes (1 et 2 deviennent 0 et 1) pour XGBoost
y = df[nom_cible] - 1

# 3. Encodage et Division Train / Test
X = pd.get_dummies(X, drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Standardisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n--- Statut ---")
print(f"Données prêtes. Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")

--- Aperçu des données ---
   age  sex   chest pain type  resting blood pressure  serum cholestoral  \
0   70     1                4                     130                322   
1   67     0                3                     115                564   
2   57     1                2                     124                261   
3   64     1                4                     128                263   
4   74     0                2                     120                269   

   fasting blood sugar  resting electrocardiographic results  max heart rate  \
0                    0                                     2             109   
1                    0                                     2             160   
2                    0                                     0             141   
3                    0                                     0             105   
4                    0                                     2             121   

   exercise induced angina  oldpeak

## Exercice 2 : Logistic Regression without Grid Search

Entraînement d'un modèle de régression logistique standard avec les paramètres par défaut de scikit-learn pour établir une première référence de performance.

In [84]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Initialisation et entraînement du modèle de base
lr_base = LogisticRegression(max_iter=1000, random_state=42)
lr_base.fit(X_train_scaled, y_train)

# Prédiction et évaluation sur l'ensemble de test
y_pred_lr = lr_base.predict(X_test_scaled)
print(f"Précision globale : {accuracy_score(y_test, y_pred_lr):.4f}")
print("\nRapport de classification complet :")
print(classification_report(y_test, y_pred_lr))

Précision globale : 0.9074

Rapport de classification complet :
              precision    recall  f1-score   support

           0       0.91      0.94      0.93        33
           1       0.90      0.86      0.88        21

    accuracy                           0.91        54
   macro avg       0.91      0.90      0.90        54
weighted avg       0.91      0.91      0.91        54



## Exercice 3 : Logistic Regression with Grid Search

Optimisation des hyperparamètres (C et penalty) de la régression logistique à l'aide de GridSearchCV pour chercher à améliorer la précision.

In [85]:
from sklearn.model_selection import GridSearchCV

# Définition de la grille des valeurs de C à tester (avec le solveur saga compatible l1 et l2)
param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2']
}

# Recherche par grille avec validation croisée
grid_lr = GridSearchCV(LogisticRegression(solver='saga', max_iter=1000, random_state=42), param_grid_lr, cv=5, scoring='accuracy', n_jobs=-1)
grid_lr.fit(X_train_scaled, y_train)

print(f"Meilleurs hyperparamètres trouvés : {grid_lr.best_params_}")
print(f"Précision sur l'ensemble de test optimisé : {grid_lr.score(X_test_scaled, y_test):.4f}")

Meilleurs hyperparamètres trouvés : {'C': 1, 'penalty': 'l2'}
Précision sur l'ensemble de test optimisé : 0.9074


c:\Users\adjara\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


## Exercice 4 : SVM without Grid Search

Entraînement d'un classificateur de machine à vecteurs de support (SVM) en configurant manuellement un noyau RBF et des paramètres standards fixes.

In [86]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Entraînement avec un noyau RBF et C=1.0 définis manuellement
svm_base = SVC(kernel='rbf', C=1.0, random_state=42)
svm_base.fit(X_train_scaled, y_train)

# Évaluation du modèle
y_pred_svm = svm_base.predict(X_test_scaled)
print(f"Précision du SVM de base : {accuracy_score(y_test, y_pred_svm):.4f}")

Précision du SVM de base : 0.8889


## Exercice 5 : SVM with Grid Search

Optimisation conjointe des hyperparamètres C, kernel et gamma du SVM par validation croisée pour trouver la meilleure frontière de décision.

In [87]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

# Grille de recherche pour le modèle SVM
param_grid_svm = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

grid_svm = GridSearchCV(SVC(random_state=42), param_grid_svm, cv=3, scoring='accuracy', n_jobs=-1)
grid_svm.fit(X_train_scaled, y_train)

print(f"Meilleurs hyperparamètres SVM : {grid_svm.best_params_}")
print(f"Précision finale du SVM optimisé sur le Test : {grid_svm.score(X_test_scaled, y_test):.4f}")

Meilleurs hyperparamètres SVM : {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}
Précision finale du SVM optimisé sur le Test : 0.8889


## Exercice 6 : XGBoost without Grid Search

Application d'un modèle d'ensemble basé sur le gradient boosting (XGBoost) en configurant manuellement ses hyperparamètres de structure.

In [88]:
!pip install xgboost


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [89]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# Initialisation avec les paramètres manuels demandés
xgb_base = XGBClassifier(learning_rate=0.1, n_estimators=100, max_depth=5, random_state=42, eval_metric='logloss')
xgb_base.fit(X_train_scaled, y_train)

# Évaluation du modèle
y_pred_xgb = xgb_base.predict(X_test_scaled)
print(f"Précision globale de XGBoost : {accuracy_score(y_test, y_pred_xgb):.4f}")

Précision globale de XGBoost : 0.8519


## Exercice 7 : XGBoost with Grid Search

Recherche exhaustive par grille sur les hyperparamètres clés de XGBoost (learning_rate, max_depth et n_estimators) afin de maximiser sa précision et sa robustesse.

In [90]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

# Définition de la grille de paramètres à explorer
param_grid_xgb = {
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5, 7],
    'n_estimators': [50, 100]
}

# Initialisation de la recherche par grille (cv=3 pour accélérer le calcul)
grid_xgb = GridSearchCV(XGBClassifier(random_state=42, eval_metric='logloss'), param_grid_xgb, cv=3, scoring='accuracy', n_jobs=-1)
grid_xgb.fit(X_train_scaled, y_train)

print(f"Meilleurs paramètres XGBoost trouvés : {grid_xgb.best_params_}")
print(f"Précision finale après optimisation : {grid_xgb.score(X_test_scaled, y_test):.4f}")

Meilleurs paramètres XGBoost trouvés : {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 50}
Précision finale après optimisation : 0.7593
